In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 1
:caption: Contents:

# Launching SWAN Cases with `swanpy`

## Imports

In [2]:
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta

from oceanicospy.models.swanpy import Initializer
from oceanicospy.models.swanpy.preprocess import GridMaker, BathyMaker, WindForcing, BottomFrictionProcessor, BoundaryConditions

## Non-stationary case:

This notebook walks through a complete SWAN **non-stationary** modelling workflow using the `swanpy` subpackage inside `oceanicospy`. The example is based on a non-stationary hindcast for the San Andres island in Colombia during May 2023.

Unlike the stationary case, a non-stationary run integrates the wave action equation over time, requiring:
* A simulation window (`ini_date` / `end_date`)
* Time-varying wind forcing (spatial ERA5 or CMDS fields)
* Time-varying water level forcing (tide gauge data)
* Time-varying boundary conditions

This scenario is also nested which means that the model is run in several domains with increasing resolution, where the output of the coarser domain is used as input for the finer domain. This allows to capture large scale wave generation processes while resolving local wave dynamics nearshore.

```{note}
The workflow assumes that input data (bathymetry `.bot`, friction `.fric`, output points `.csv`, and tide-gauge CSV) are already placed in the appropriate `input/domain_0X/` folder inside the case directory. ERA5 wind and wave data are downloaded automatically if not already present.

### Case parameters definition

Four parameters drive the non-stationary setup:

| Parameter | Description |
|---|---|
| `path_case` | Root directory holding `input/`, `run/`, `output/`, `pros/` |
| `dict_ini_data` | Metadata dictionary consumed by `Initializer` |
| `ini_date` | Start of the simulation window |
| `end_date` | End of the simulation window |

The `dict_ini_data` dictionary accepts the following keys:

- `name`: A string identifier for the case.
- `case_number` (optional): Integer to differentiate between multiple cases.
- `case_description`: Brief description of the case.
- `stat_id`: `0` for non-stationary, `1` for stationary.
- `number_domains`: Total number of computational domains.
- `parent_domains`: Dict mapping each domain id to its parent domain id (or `None` for top-level domains). Required for `write_nest_section()` to correctly handle nesting output.

#### Domain iteration workflow

All the definition of input data and parameters will be done using an iteration over the domains, which is a common workflow for nested cases. This allows to easily scale up to any number of domains and correctly link parent-child relationships for nesting. First, we define a dictionary with the necessary information for each domain, and then we loop through it to initialize the case and populate the `run.swn` files.

In [3]:
domains = {
    1: {
        "grid": dict(lon_ll_corner=278.2, lat_ll_corner=12.4, x_extent=0.2, y_extent=0.3, nx=222, ny=333),
        "bathy": dict(lon_ll_corner_bot=278.1811, lat_ll_corner_bot=12.372, nx_bot=322, ny_bot=422, dx_bot=0.0009, dy_bot=0.0009),
        "fric": dict(lon_ll_corner_fric=278.1811, lat_ll_corner_fric=12.372, nx_fric=322, ny_fric=422, dx_fric=0.0009, dy_fric=0.0009),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=278.1811, lat_ll_corner_wl=12.372, nx_wl=322, ny_wl=422, dx_wl=0.0009, dy_wl=0.0009),
        "parent": None
    },
    2: {
        "grid": dict(lon_ll_corner=278.2265167, lat_ll_corner=12.4413166, x_extent=0.13185, y_extent=0.1908, nx=293, ny=424),
        "bathy": dict(lon_ll_corner_bot=278.2265167, lat_ll_corner_bot=12.4413166, nx_bot=293, ny_bot=424, dx_bot=0.00045, dy_bot=0.00045),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=278.2265167, lat_ll_corner_wl=12.4413166, nx_wl=293, ny_wl=424, dx_wl=0.00045, dy_wl=0.00045),
        "fric": dict(lon_ll_corner_fric=278.2265167, lat_ll_corner_fric=12.4413166, nx_fric=293, ny_fric=424, dx_fric=0.00045, dy_fric=0.00045),
        "parent": 1
    },
    3: {
        "grid": dict(lon_ll_corner=278.270165, lat_ll_corner=12.550217, x_extent=0.0747, y_extent=0.0747, nx=332, ny=332),
        "bathy": dict(lon_ll_corner_bot=278.270165, lat_ll_corner_bot=12.550217, nx_bot=332, ny_bot=332, dx_bot=0.000225, dy_bot=0.000225),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=278.270165, lat_ll_corner_wl=12.550217, nx_wl=332, ny_wl=332, dx_wl=0.000225, dy_wl=0.000225),
        "fric": dict(lon_ll_corner_fric=278.270165, lat_ll_corner_fric=12.550217, nx_fric=332, ny_fric=332, dx_fric=0.000225, dy_fric=0.000225),
        "parent": 2
    },
    4: {
        "grid": dict(lon_ll_corner=360 - 81.72, lat_ll_corner=12.5, x_extent=0.055, y_extent=0.055, nx=243, ny=243),
        "bathy": dict(lon_ll_corner_bot=360 - 81.72, lat_ll_corner_bot=12.5, nx_bot=243, ny_bot=243, dx_bot=0.000225, dy_bot=0.000225),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=360 - 81.72, lat_ll_corner_wl=12.5, nx_wl=243, ny_wl=243, dx_wl=0.000225, dy_wl=0.000225),
        "fric": dict(lon_ll_corner_fric=360 - 81.72, lat_ll_corner_fric=12.5, nx_fric=243, ny_fric=243, dx_fric=0.000225, dy_fric=0.000225),
        "parent": 2
    }
}

Each key in the `domains` dictionary corresponds to a domain id, and the value is another dictionary with the following keys:
- `grid`: Dict with grid parameters (`x_min`, `x_max`, `y_min`, `y_max`, `resolution`).
- `bathy`: Dict with spatial bathymetry parameters.
- `wind`: Dict with spatial wind parameters.
- `wl`: Dict with spatial water level parameters.
- `fric`: Dict with spatial friction parameters.
- `parent`: Parent domain id (or `None` for top-level domains).


In [4]:
path_case = '../data/model_runs/nonstationary_swan/'

ini_date = datetime(2023, 5, 7, 0)
end_date = datetime(2023, 5, 31, 23)

dict_ini_data = dict(
    name             = 'Non-stationary case for San Andres',
    case_number      = 2,
    case_description = 'Example non-stat',
    stat_id          = 0, # 0 → non-stationary
    number_domains   = len(domains),
    parent_domains   = {d:cfg["parent"] for d, cfg in domains.items()} # domain 1 has no parent and domain 3 and 4 share parent domain 2.
)

```{hint}
`stat_id` convention
* `stat_id = 0` → **non-stationary** run  ← _used in this notebook_
* `stat_id = 1` → **stationary** run

### Initial setup

`Initializer` creates the project directory tree and writes the baseline SWAN configuration file (`run.swn`) from the **non-stationary template** (`run_base_nonstat.swn`). The directory layout is:

```
path_case/
├── input/
│   └── domain_01/          ← place ERA5 .nc, .bot, .fric and tide-gauge CSV here
│   └── domain_02/ 
│   └── domain_03/ 
│   └── domain_04/ 
├── pros/
├── run/
│   └── domain_01/
│       └── run.swn         ← generated from non-stationary template
│   └── domain_02/
│       └── run.swn
│   └── domain_03/
│       └── run.swn
│   └── domain_04/
│       └── run.swn
└── output/
    └── domain_01/
    └── domain_02/
    └── domain_03/
    └── domain_04/
```

```{note}
Pass `ini_date` and `end_date` to `Initializer` for non-stationary runs. These are stored in `case.ini_date` and `case.end_date` and used by modules like `WindForcing`, `WaterLevelForcing`, and `BoundaryConditions` to slice the downloaded time series to the simulation window.

In [5]:
case = Initializer(
    root_path     = path_case,
    dict_ini_data = dict_ini_data,
    ini_date      = ini_date,
    end_date      = end_date,
)

case.create_folders()       # creates all project folders
case.replace_ini_data()     # writes run.swn from the NONSTAT template

*** Initializing SWAN model ***


	*** Creating project folder structure ***


	*** Copying base SWAN configuration file into run folder ***



### Creating the grid

`GridMaker` populates the `CGRID` section of `run.swn` for every computational domain. We'll reach this using that method under an iteration over the domains.

In [6]:
for domain_id,config in domains.items():
    domain_grid_dict = config["grid"]
    case_grid = GridMaker(init = case,
                          domain_number = domain_id, 
                          dict_info = domain_grid_dict)
    case_grid.fill_grid_section()


*** Initializing gridmaker for domain 1 ***


 	*** Adding/Editing grid information for domain 1 in configuration file ***


*** Initializing gridmaker for domain 2 ***


 	*** Adding/Editing grid information for domain 2 in configuration file ***


*** Initializing gridmaker for domain 3 ***


 	*** Adding/Editing grid information for domain 3 in configuration file ***


*** Initializing gridmaker for domain 4 ***


 	*** Adding/Editing grid information for domain 4 in configuration file ***



### Setting up the bathymetry

`BathyMaker` fills the `BOTTOM` section of each `run.swn` for each domain. Use `use_ascii_file_from_user()` when a pre-processed `.bot` file already exists in `input/domain_XX/`.

```{note}
An iteration is made for every type of input data only for illustration purposes. In practice, a single iteration can be made where all the necessary methods are called for each domain, which is more efficient. The iteration is separated here for clarity.


In [7]:
for domain_id,config in domains.items():
    domain_bathy_dict = config["bathy"]
    case_bathy = BathyMaker(init = case,
                          domain_number = domain_id, 
                          dict_info = domain_bathy_dict)
    case_bathy.use_ascii_file_from_user()
    case_bathy.fill_bathy_section()


*** Initializing bathymaker for domain 1 ***


 	*** Adding/Editing bathymetry information for domain 1 in configuration file ***


*** Initializing bathymaker for domain 2 ***


 	*** Adding/Editing bathymetry information for domain 2 in configuration file ***


*** Initializing bathymaker for domain 3 ***


 	*** Adding/Editing bathymetry information for domain 3 in configuration file ***


*** Initializing bathymaker for domain 4 ***


 	*** Adding/Editing bathymetry information for domain 4 in configuration file ***



### Defining bottom friction information

`BottomFrictionProcessor` is a class inside `swanpy` that fills the `FRICTION` section of `run.swn` with the appropriate parameters for bottom friction.

In [8]:
for domain_id,config in domains.items():
    domain_bathy_dict = config["fric"]
    case_bathy = BottomFrictionProcessor(init = case,
                          domain_number = domain_id, 
                          dict_info = domain_bathy_dict)
    case_bathy.use_ascii_file_from_user()
    case_bathy.fill_friction_section()


*** Initializing BottomFrictionProcessor for domain 1 ***


 	*** Adding/Editing friction information for domain 1 in configuration file ***


*** Initializing BottomFrictionProcessor for domain 2 ***


 	*** Adding/Editing friction information for domain 2 in configuration file ***


*** Initializing BottomFrictionProcessor for domain 3 ***


 	*** Adding/Editing friction information for domain 3 in configuration file ***


*** Initializing BottomFrictionProcessor for domain 4 ***


 	*** Adding/Editing friction information for domain 4 in configuration file ***



### Defining wind forcing from ERA5

For a non-stationary run, SWAN requires a **spatially distributed, time-varying** wind field rather than a single constant vector. `WindForcing` handles:

1. **Downloading** ERA5 (or CMDS) 10-m wind components for the simulation window.
2. **Converting** the NetCDF output to SWAN's binary-equivalent ASCII format.
3. **Writing** the `INP WIND` / `READ WIND` section of `run.swn`.

The spatial grid for the wind field is specified through `dict_info`. It does not need to match the computational grid resolution, but must cover the entire domain.

```{note}
When `share_winds=True` (the default), domain 1 downloads the wind data and subsequent domains reuse it, which is the recommended approach for nested multi-domain cases.

In [12]:
for domain_id,config in domains.items():
    domain_wind_dict = config["wind"]
    case_wind = WindForcing(init = case,
                          domain_number = domain_id, 
                          dict_info = domain_wind_dict)
    # Skips download if the file already exists in input/domain_01/
    case_wind.get_winds_from_ERA5(difference_to_UTC=-5)

    # Convert NetCDF → SWAN ASCII and link into run/domain_01/
    case_wind.write_ERA5_ascii('winds_era5.nc', 'winds.wnd')
    case_wind.fill_wind_section()


*** Initializing winds for domain 1 ***



2026-04-09 14:42:12,225 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-09 14:42:12,225 INFO Request ID is 343b4428-ce1b-47d2-a717-384fc89add5f
2026-04-09 14:42:12,443 INFO status has been updated to accepted
2026-04-09 14:42:21,473 INFO status has been updated to running
2026-04-09 14:45:08,139 INFO status has been updated to successful


2862e6a0a159aa7c2b4a611f3d084588.nc:   0%|          | 0.00/2.79M [00:00<?, ?B/s]

Downloaded winds_era5.nc to /Users/franklinayala/Documents/projects/oceanicospy/data/model_runs/nonstationary_swan/input/domain_01/winds_era5.nc.
	 ERA5 wind data downloaded successfully
	 ERA5 wind data converted to ASCII format and saved as winds.wnd

 	*** Adding/Editing winds information for domain 1 in configuration file ***


*** Initializing winds for domain 2 ***

	 ERA5 wind data already exists in domain 1, skipping download
	 ERA5 wind data converted to ASCII format and saved as winds.wnd in domain 01, linking to domain 2

 	*** Adding/Editing winds information for domain 2 in configuration file ***


*** Initializing winds for domain 3 ***

	 ERA5 wind data already exists in domain 1, skipping download
	 ERA5 wind data converted to ASCII format and saved as winds.wnd in domain 01, linking to domain 3

 	*** Adding/Editing winds information for domain 3 in configuration file ***


*** Initializing winds for domain 4 ***

	 ERA5 wind data already exists in domain 1, skipping d

```{hint}
To use CMDS wind data instead of ERA5, replace `get_winds_from_ERA5` / `write_ERA5_ascii` with
`get_winds_from_CMDS` / `write_CMDS_ascii`. The rest of the workflow is identical.

### Defining water level forcing

`WaterLevelForcing` handles tide-gauge water level data from UHSLC (University of Hawaii Sea Level Center):

1. **Downloading** hourly tide-gauge data for the simulation window from UHSLC.
2. **Converting** the CSV to a spatially uniform SWAN ASCII water-level file (one grid snapshot per timestep).
3. **Writing** the `INP WLEV` / `READ WLEV` section of `run.swn`.

The San Andres UHSLC station code is **737**.

In [ ]:
dict_info_wl = dict(
    lon_ll_wl   = 278.1811,
    lat_ll_wl   = 12.372,
    meshes_x_wl = 322,
    meshes_y_wl = 422,
    dx_wl       = 0.0009,
    dy_wl       = 0.0009,
)

case_wl = WaterLevelForcing(
    init          = case,
    domain_number = DOMAIN_ID,
    wl_info       = dict_info_wl,
    share_wl      = True,
    use_link      = True,
)

# Download UHSLC hourly tide-gauge record (skips if h737.csv already present)
case_wl.get_waterlevel_from_UHSLC(station_code=737)

# Convert CSV → SWAN ASCII and link into run/domain_01/
dict_wl = case_wl.write_UHSLC_ascii('h737.csv', 'water_levels.wl')

print(f'Water level info for domain {DOMAIN_ID}: {dict_wl}')
case_wl.fill_wl_section(dict_wl)

### Defining boundary conditions

For non-stationary runs, SWAN supports two approaches to spectral boundary conditions:

| Approach | SWAN command | When to use |
|---|---|---|
| **Constant** (`CON PAR`) | `BOUN SIDE N CLOCKW CON PAR Hs Tp Dir dd` | Quick tests, swell climatology |
| **Time-varying** (`VAR FILE`) | `BOUN SIDE N CLOCKW VAR FILE …` | Calibration, hindcasts requiring realistic offshore spectra |

#### Option A – Constant spectral boundary conditions (current API)

Use `create_boundary_line()` + `fill_boundaries_section()` to write a constant parametric spectrum on one or more sides. This approach works for both stationary and non-stationary runs.

In [ ]:
case_boundary = BoundaryConditions(
    init          = case,
    domain_number = DOMAIN_ID,
    dict_info     = {'boundary_type': 'side'},
)

case_boundary.create_boundary_line(
    list_sides  = ['N'],
    wave_params = {'hs': 1.5, 'tp': 10, 'dir': 90, 'spread': 2}
)
case_boundary.fill_boundaries_section()

#### Option B – Time-varying TPAR boundary conditions from ERA5

For calibration runs or realistic hindcasts, use time-varying spectra extracted from ERA5 (or CMDS) wave reanalysis at a set of boundary points. The workflow is:

1. `get_waves_from_ERA5()` – download ERA5 wave parameters (Hs, Tp, MWD) for the region.
2. `tpar_from_ERA5()` – extract point time series at the boundary corners and write individual `.bnd` TPAR files.

After generating the TPAR files, add the corresponding `BOUN SIDE VAR FILE` commands to `run.swn` following SWAN syntax.

In [ ]:
# Boundary point arrays — define the lat/lon grid of boundary sample points
lat_boundaries = [12.40, 12.50, 12.60, 12.70]
lon_boundaries = [-81.80, -81.70, -81.60]   # negative longitudes are wrapped to [0, 360) internally

case_boundary_tpar = BoundaryConditions(
    init          = case,
    domain_number = DOMAIN_ID,
    dict_info     = {'boundary_type': 'side'},
    use_link      = True,
)

# Step 1 – download ERA5 wave parameters (Hs, Tp, MWD) for the boundary region
case_boundary_tpar.get_waves_from_ERA5(
    difference_to_UTC = -5,
    wind_info_dict    = dict_info_wind,
)

# Step 2 – extract time series at boundary corners and write .bnd TPAR files
case_boundary_tpar.tpar_from_ERA5(
    points_lat = lat_boundaries,
    points_lon = lon_boundaries,
)

```{note}
After calling `tpar_from_ERA5()`, the `.bnd` files are generated in `input/domain_01/` and linked into the run directory. To activate them in SWAN, add the `BOUN SIDE VAR FILE` commands to `run.swn` following SWAN manual §4.5 syntax — for example:

```
BOUN SIDE N CLOCKW VAR FILE  0 '../../input/domain_01/TparN2025_1.bnd' 1 &
                             0.1 '../../input/domain_01/TparN2025_2.bnd' 1 &
                             0.2 '../../input/domain_01/TparN2025_3.bnd' 1
```
Full TPAR fill automation via `fill_boundaries_section()` is under development.

### Defining physics

`PhysicsMaker` configures the wave generation and dissipation scheme via the `GEN3 WESTIN` command. The two tunable parameters are:

| Parameter | Symbol | Description |
|---|---|---|
| `cds1` | $C_{ds1}$ | Whitecapping dissipation coefficient |
| `delta` | $\delta$ | Cumulative effect of breaking coefficient |

In [ ]:
case_physics = PhysicsMaker(
    init          = case,
    domain_number = DOMAIN_ID,
)

dict_physics = case_physics.define_generation(cds1=6.3, delta=0.5)

print(f'Physics parameters: {dict_physics}')
case_physics.fill_physics_section(dict_physics)

### Computation and run configuration

`CaseRunner` finalises the configuration. For a non-stationary run it:

* `write_nest_section()` – removes unused `NESTOUT`/`NGRID` lines for a single-domain case, or fills in nest-output commands for multi-domain cases.
* `define_output_from_file()` – reads a CSV of output point coordinates and writes `points.loc`.
* `fill_computation_section()` – writes `COMP NONSTAT ini_date dt_min MIN end_date` into `run.swn`.

The non-stationary computation dictionary (`dict_comp_data`) requires:

| Key | Description |
|---|---|
| `stat_comp` | Always `1` (selects the computation string builder) |
| `ini_comp_date` | Start date of the computation (format `YYYYMMDD.HHMMSS`) |
| `dt_min` | Time step in minutes |
| `end_comp_date` | End date of the computation |
| `ini_output_date` | First output dump date (allows a spin-up period before saving) |
| `output_res_min` | Output interval in minutes |

In [ ]:
comp_data_nonstat = dict(
    stat_comp       = 1,
    ini_comp_date   = ini_date.strftime('%Y%m%d.%H%M%S'),
    dt_min          = 10,
    end_comp_date   = end_date.strftime('%Y%m%d.%H%M%S'),
    ini_output_date = (ini_date + timedelta(days=5)).strftime('%Y%m%d.%H%M%S'),  # 5-day spin-up
    output_res_min  = 60,
)

In [ ]:
case_run = CaseRunner(
    init           = case,
    domain_number  = DOMAIN_ID,
    dict_comp_data = comp_data_nonstat,
    all_domains    = None,            # set to domains dict for multi-domain cases
)

# Remove NESTOUT/NGRID template lines (single-domain has no child domains)
case_run.write_nest_section()

# Write output point locations from CSV (X, Y columns)
case_run.define_output_from_file(filename='output_points.csv')

# Write the COMP NONSTAT command and output date/interval placeholders
case_run.fill_computation_section()

# (Optional) generate SLURM launcher for HPC submission
# case_run.fill_slurm_file()

All configuration is now written to `run/domain_01/run.swn`. The case is ready to be executed with the SWAN binary locally or submitted to an HPC scheduler.

### Multi-domain nested case

The workflow above covers a single-domain run. For a **nested multi-domain** case (e.g., four progressively finer grids as in `examples/main_swan_nonstat.py`), define a domain configuration dictionary and iterate over domains:

```python
domains = {
    1: {
        "grid" : dict(lon_ll_corner=278.2, lat_ll_corner=12.4, x_extent=0.2,  y_extent=0.3,  nx=222, ny=333),
        "bathy": dict(lon_ll_corner_bot=278.1811, lat_ll_corner_bot=12.372,   nx_bot=322, ny_bot=422, dx_bot=0.0009, dy_bot=0.0009),
        "wind" : dict(lon_ll_wind=278, lat_ll_wind=12.3, meshes_x_wind=24, meshes_y_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl"   : dict(lon_ll_wl=278.1811, lat_ll_wl=12.372, meshes_x_wl=322, meshes_y_wl=422, dx_wl=0.0009, dy_wl=0.0009),
        "fric" : dict(lon_ll_corner_fric=278.1811, lat_ll_corner_fric=12.372, nx_fric=322,    ny_fric=422,  dx_fric=0.0009, dy_fric=0.0009),
        "parent": None,
    },
    2: {
        "grid" : dict(lon_ll_corner=278.2265167, lat_ll_corner=12.4413166, x_extent=0.13185, y_extent=0.1908, nx=293, ny=424),
        "bathy": dict(lon_ll_corner_bot=278.2265167, lat_ll_corner_bot=12.4413166, nx_bot=293, ny_bot=424, dx_bot=0.00045, dy_bot=0.00045),
        "wind" : dict(lon_ll_wind=278, lat_ll_wind=12.3, meshes_x_wind=24, meshes_y_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl"   : dict(lon_ll_wl=278.2265167, lat_ll_wl=12.4413166, meshes_x_wl=293, meshes_y_wl=424, dx_wl=0.00045, dy_wl=0.00045),
        "fric" : dict(lon_ll_corner_fric=278.2265167, lat_ll_corner_fric=12.4413166, nx_fric=293, ny_fric=424, dx_fric=0.00045, dy_fric=0.00045),
        "parent": 1,
    },
    # … add domain 3, 4 following the same pattern
}

dict_ini_data_multi = dict(
    name='May 2023 San Andres', case_number=6,
    case_description='Calibration', stat_id=0,
    number_domains=2,
    parent_domains={d: cfg['parent'] for d, cfg in domains.items()}
)

case = Initializer(root_path=path_case, dict_ini_data=dict_ini_data_multi, ini_date=ini_date, end_date=end_date)
case.create_folders()
case.replace_ini_data()

for domain_id, config in domains.items():
    # Grid
    GridMaker(init=case, domain_number=domain_id, dict_info=config['grid']).fill_grid_section()
    # Bathymetry
    bm = BathyMaker(init=case, domain_number=domain_id, dict_info=config['bathy'])
    bm.use_ascii_file_from_user(); bm.fill_bathy_section()
    # Wind (domain 1 downloads; others share)
    wf = WindForcing(init=case, domain_number=domain_id, dict_info=config['wind'], share_winds=True, use_link=True)
    wf.get_winds_from_ERA5(difference_to_UTC=-5)
    wf.write_ERA5_ascii('winds_era5.nc', 'winds.wnd'); wf.fill_wind_section()
    # Water level (domain 1 downloads; others share)
    wl = WaterLevelForcing(init=case, domain_number=domain_id, wl_info=config['wl'], share_wl=True, use_link=True)
    wl.get_waterlevel_from_UHSLC(station_code=737)
    dict_wl = wl.write_UHSLC_ascii('h737.csv', 'water_levels.wl'); wl.fill_wl_section(dict_wl)
    # Friction
    fr = BottomFrictionProcessor(init=case, domain_number=domain_id, dict_info=config['fric'])
    fr.use_ascii_file_from_user(); fr.fill_friction_section()
    # Physics
    ph = PhysicsMaker(init=case, domain_number=domain_id)
    ph.fill_physics_section(ph.define_generation(cds1=6.3, delta=0.5))
    # Computation
    comp_data = dict(stat_comp=1,
                     ini_comp_date=ini_date.strftime('%Y%m%d.%H%M%S'),
                     dt_min=10,
                     end_comp_date=end_date.strftime('%Y%m%d.%H%M%S'),
                     ini_output_date=(ini_date + timedelta(days=5)).strftime('%Y%m%d.%H%M%S'),
                     output_res_min=60)
    runner = CaseRunner(init=case, domain_number=domain_id, dict_comp_data=comp_data, all_domains=domains)
    runner.define_output_from_file(filename='output_points.csv')
    runner.write_nest_section()
    runner.fill_computation_section()
```

### Reading the output

After running the case, the output files will be generated in the `output/domain_01/` folder and can be post-processed using the `swanpy` post-processing utilities.

In [ ]:
  _common = dict(                                                                                                                   
      ini_output_date = (ini_date + timedelta(days=5)).strftime('%Y%m%d.%H%M%S'),                                                     
      output_res_min  = 60,
      stat_comp       = 1,                                                                                                            
      dt_min          = 10,                                                                                                         
      end_comp_date   = end_date.strftime('%Y%m%d.%H%M%S'),                                                                           
  )                                                                                                                                   
  compilation_data = {
      1: {**_common, "ini_comp_date": ini_date.strftime('%Y%m%d.%H%M%S'),                                                             
                     "ini_nest_date": (ini_date + timedelta(days=2)).strftime('%Y%m%d.%H%M%S')},                                      
      2: {**_common, "ini_comp_date": (ini_date + timedelta(days=2)).strftime('%Y%m%d.%H%M%S'),                                       
                     "ini_nest_date": (ini_date + timedelta(days=2)).strftime('%Y%m%d.%H%M%S')},                                      
      3: {**_common, "ini_comp_date": (ini_date + timedelta(days=2)).strftime('%Y%m%d.%H%M%S')},                                      
      4: {**_common, "ini_comp_date": (ini_date + timedelta(days=2)).strftime('%Y%m%d.%H%M%S')},                                      
  }  